# Deep Drowsiness Detection — CNN-GRU & TSM (+ MAR features)

Training, calibration, and evaluation of the deep video models for the ES327 driver-drowsiness study.

Two temporal architectures share a ResNet-18 backbone and are fused with a per-clip **Mouth-Aspect-Ratio (MAR)** feature vector:

- **CNN-GRU** — ResNet-18 per-frame features → GRU (primary model; best external/DROZY performance)
- **TSM-fast** — ResNet-18 with temporal averaging (lightweight baseline)

R(2+1)D-18 is trained and evaluated in a separate notebook.

**Pipeline:** Setup → MAR cache → Model builders → Train → Calibrate (freeze threshold) → Evaluate on UTA-RLDD / YawDD / DROZY.

**Final checkpoint:** `cnn_gru_mar_final.pt` (weights + temperature + F1-selected threshold τ≈0.001).

---
## 1. Setup & Data Preparation

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

In [ ]:
import os
import pandas as pd
from collections import Counter

SPLITS_DIR = "/content/drive/MyDrive/ES327_Drowsiness/splits"

UTA_TRAIN  = os.path.join(SPLITS_DIR, "uta_train.csv")
UTA_VAL    = os.path.join(SPLITS_DIR, "uta_val.csv")
YAW_TRAIN  = os.path.join(SPLITS_DIR, "yawdd_train.csv")
YAW_VAL    = os.path.join(SPLITS_DIR, "yawdd_val.csv")

def load_split_csv(path):
    df = pd.read_csv(path)
    df = df.dropna(subset=["clip_path", "label"]).copy()
    df["label"] = df["label"].astype(int)
    return df

def namespace_subjects(df, default_ds):
    df = df.copy()
    if "dataset" in df.columns:
        ds = df["dataset"].fillna(default_ds).astype(str).str.lower()
    else:
        ds = default_ds
    df["subject_id"] = ds + "_" + df["subject_id"].astype(str)
    return df

uta_train = namespace_subjects(load_split_csv(UTA_TRAIN), "uta")
uta_val   = namespace_subjects(load_split_csv(UTA_VAL),   "uta")
yaw_train = namespace_subjects(load_split_csv(YAW_TRAIN), "yawdd")
yaw_val   = namespace_subjects(load_split_csv(YAW_VAL),   "yawdd")

train_df = pd.concat([uta_train, yaw_train], ignore_index=True)
val_df   = pd.concat([uta_val,   yaw_val],   ignore_index=True)

print("Train:", len(train_df), dict(Counter(train_df["label"])))
print("Val:", len(val_df), dict(Counter(val_df["label"])))

In [ ]:
import os
import pandas as pd

CLIPS_DRIVE = "/content/drive/MyDrive/ES327_Drowsiness/Clips"
CLIPS_LOCAL = "/content/Clips"

# train_df + val_df already loaded above (must have clip_path)
all_paths = pd.concat([train_df["clip_path"], val_df["clip_path"]]).dropna().astype(str).unique()

prefix = CLIPS_DRIVE.rstrip("/") + "/"
rel_paths = [p[len(prefix):] for p in all_paths if p.startswith(prefix)]

print("Train+Val unique clips:", len(rel_paths))

filelist = "/content/needed_trainval.txt"
with open(filelist, "w") as f:
    f.write("\n".join(rel_paths))

print("✅ Wrote:", filelist)
print("Example:", rel_paths[0])

In [ ]:
filelist = "/content/needed_trainval.txt"
bad_sub = "Fold3_part1__29__10__00122.npz"  # match by substring

with open(filelist, "r") as f:
    lines = [ln.strip() for ln in f if ln.strip()]

kept = [ln for ln in lines if bad_sub not in ln]

with open(filelist, "w") as f:
    f.write("\n".join(kept) + "\n")

print("List size before:", len(lines))
print("List size after :", len(kept))
print("Removed         :", len(lines) - len(kept))

In [ ]:
!mkdir -p /content/Clips

!rsync -ah \
  --info=progress2 \
  --partial \
  --ignore-existing \
  --ignore-missing-args \
  --ignore-errors \
  --files-from=/content/needed_trainval.txt \
  "/content/drive/MyDrive/ES327_Drowsiness/Clips/" \
  "/content/Clips/"

---
## 2. MAR Feature Cache (FaceMesh)

Builds the per-clip Mouth-Aspect-Ratio cache used as an auxiliary feature by both models. **Optional:** the training script in Section 4 also builds/resumes this cache automatically if it is missing, so you only need to run this cell if you want to pre-build the cache on its own.

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os, json, time
import numpy as np
import pandas as pd
from tqdm import tqdm
import mediapipe as mp

# ----------------------------
# Paths
# ----------------------------
SPLITS = [
    "/content/drive/MyDrive/ES327_Drowsiness/splits/uta_train.csv",
    "/content/drive/MyDrive/ES327_Drowsiness/splits/uta_val.csv",
    "/content/drive/MyDrive/ES327_Drowsiness/splits/yawdd_train.csv",
    "/content/drive/MyDrive/ES327_Drowsiness/splits/yawdd_val.csv",
]

SAVE_DIR = "/content/drive/MyDrive/ES327_Drowsiness/outputs_mar_cache"
os.makedirs(SAVE_DIR, exist_ok=True)
MAR_CACHE_PATH = os.path.join(SAVE_DIR, "mar_cache.json")

# ----------------------------
# Settings
# ----------------------------
MAR_FRAMES = 4
MAR_THRESHOLD = 0.55
SAVE_EVERY = 200

# Mouth indices (MediaPipe FaceMesh)
MOUTH_L, MOUTH_R, MOUTH_U, MOUTH_D = 61, 291, 13, 14

def clean_path(p: str) -> str:
    return str(p).replace("\\", "/").replace("\u00a0"," ").strip()

def load_split_csv(path):
    df = pd.read_csv(path)
    df = df.dropna(subset=["clip_path", "label"]).copy()
    df["clip_path"] = df["clip_path"].map(clean_path)
    return df

def sample_indices(T: int, n: int) -> np.ndarray:
    if T <= 0: return np.array([], dtype=int)
    if n <= 1: return np.array([T//2], dtype=int)
    return np.linspace(0, T-1, n).round().astype(int)

def clip_to_rgb_frames(npz_path: str) -> np.ndarray:
    """
    Loads npz['clip'] shaped (C,T,H,W) = (3,16,112,112)
    Returns uint8 RGB frames [T,H,W,3]
    """
    z = np.load(npz_path, allow_pickle=True)
    x = z["clip"] if "clip" in z else z[z.files[0]]

    # x: (C,T,H,W) -> (T,H,W,C)
    if x.ndim != 4:
        raise ValueError(f"Unexpected ndim {x.ndim} keys={z.files}")

    # Detect (C,T,H,W)
    if x.shape[0] == 3:
        x = np.transpose(x, (1, 2, 3, 0))   # -> (T,H,W,C)
    # Detect (T,C,H,W)
    elif x.shape[1] == 3:
        x = np.transpose(x, (0, 2, 3, 1))   # -> (T,H,W,C)
    # Detect already (T,H,W,C)
    elif x.shape[-1] == 3:
        pass
    else:
        raise ValueError(f"Unknown clip layout {x.shape} in {npz_path}")

    # to uint8 RGB
    x = x.astype(np.float32)
    if x.max() <= 1.5:
        x *= 255.0
    x = np.clip(x, 0, 255).astype(np.uint8)
    return x  # [T,H,W,3]

def mar_from_landmarks(lms, h: int, w: int) -> float:
    def pt(i):
        lm = lms[i]
        return np.array([lm.x * w, lm.y * h], dtype=np.float32)
    L = pt(MOUTH_L); R = pt(MOUTH_R); U = pt(MOUTH_U); D = pt(MOUTH_D)
    horiz = np.linalg.norm(L - R) + 1e-6
    vert  = np.linalg.norm(U - D)
    return float(vert / horiz)

def compute_features(frames_rgb: np.ndarray, face_mesh) -> list:
    T, H, W, _ = frames_rgb.shape
    idxs = sample_indices(T, MAR_FRAMES)
    mars = []
    for i in idxs:
        res = face_mesh.process(frames_rgb[i])  # RGB
        if not res.multi_face_landmarks:
            continue
        lms = res.multi_face_landmarks[0].landmark
        mars.append(mar_from_landmarks(lms, H, W))

    if len(mars) == 0:
        return [0.0, 0.0, 0.0, 0.0]

    mars = np.array(mars, dtype=np.float32)
    return [
        float(mars.mean()),
        float(mars.max()),
        float(mars.std()),
        float((mars > MAR_THRESHOLD).mean())
    ]

# ----------------------------
# Build cache
# ----------------------------
dfs = [load_split_csv(s) for s in SPLITS if os.path.exists(s)]
all_df = pd.concat(dfs, ignore_index=True)
clip_paths = all_df["clip_path"].unique().tolist()
print("Unique clips:", len(clip_paths))

# Resume cache
if os.path.exists(MAR_CACHE_PATH):
    mar_cache = json.load(open(MAR_CACHE_PATH))
    mar_cache = {clean_path(k): v for k,v in mar_cache.items()}
    print("Resuming entries:", len(mar_cache))
else:
    mar_cache = {}

mp_face_mesh = mp.solutions.face_mesh

t0 = time.time()
added, no_face, errors = 0, 0, 0

with mp_face_mesh.FaceMesh(
    static_image_mode=True,
    max_num_faces=1,
    refine_landmarks=False,
    min_detection_confidence=0.5,
) as face_mesh:

    for p in tqdm(clip_paths):
        key = clean_path(p)
        if key in mar_cache:
            continue

        try:
            frames = clip_to_rgb_frames(key)
            feats = compute_features(frames, face_mesh)
            if feats == [0.0, 0.0, 0.0, 0.0]:
                no_face += 1
            mar_cache[key] = feats
            added += 1

            if added % SAVE_EVERY == 0:
                json.dump(mar_cache, open(MAR_CACHE_PATH, "w"))

        except Exception:
            errors += 1
            if errors <= 3:
                import traceback
                print("First error on:", key)
                traceback.print_exc()
            continue

json.dump(mar_cache, open(MAR_CACHE_PATH, "w"))

dt = (time.time() - t0) / 60
print("\n✅ Done")
print("Saved:", MAR_CACHE_PATH)
print("Total entries:", len(mar_cache))
print("Newly added:", added)
print("No-face:", no_face)
print("Errors:", errors)
print(f"Time: {dt:.1f} min")

---
## 3. Model Builders

Defines all three architectures (3D-ResNet-18, CNN-GRU, TSM-fast). CNN-GRU and TSM are used here; the 3D-CNN builder is included for completeness.

In [ ]:
import torchvision
import torch.nn as nn

NUM_CLASSES = 2

# 3D-CNN: 3D ResNet-18
def build_r3d18(num_classes=2):
    m = torchvision.models.video.r3d_18(weights=None)
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m

# CNN-GRU: ResNet18 per-frame + GRU
class ResNetGRU(nn.Module):
    def __init__(self, hidden=128, num_classes=2):
        super().__init__()
        base = torchvision.models.resnet18(weights=None)
        self.backbone = nn.Sequential(*list(base.children())[:-1])
        self.feat_dim = 512
        self.gru = nn.GRU(self.feat_dim, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, num_classes)

    def forward(self, x):  # (B,T,C,H,W)
        B,T,C,H,W = x.shape
        x = x.reshape(B*T, C, H, W)
        f = self.backbone(x).flatten(1)
        f = f.reshape(B, T, self.feat_dim)
        out, _ = self.gru(f)
        return self.fc(out[:, -1, :])

def build_cnn_gru(num_classes=2):
    # ✅ MUST be 128 to match your saved checkpoint
    return ResNetGRU(hidden=128, num_classes=num_classes)

# TSM-style temporal average baseline
class ResNetTemporalAvg(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        base = torchvision.models.resnet18(weights=None)
        self.backbone = nn.Sequential(*list(base.children())[:-1])
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):  # (B,C,T,H,W)
        B,C,T,H,W = x.shape
        x = x.permute(0,2,1,3,4).contiguous().view(B*T, C, H, W)
        f = self.backbone(x).flatten(1).view(B, T, 512).mean(dim=1)
        return self.fc(f)

def build_tsm_fast(num_classes=2):
    return ResNetTemporalAvg(num_classes=num_classes)

print("✅ Builders defined.")
print("Sanity check hidden:", build_cnn_gru().gru.hidden_size)

---
## 4. Training (CNN-GRU + TSM, with MAR fusion)

End-to-end training script. Builds/resumes the MAR cache, trains both TSM-fast (primary) and CNN-GRU, calibrates on the global validation set (temperature scaling + threshold selection), and saves `*_best.pt` checkpoints. Resumable across Colab interruptions.

In [ ]:
# ============================================================
# FINAL BIG SCRIPT (WORKS WITH YOUR BUILDERS + YOUR NPZ FORMAT)
#
# Assumptions confirmed:
# - NPZ key: "clip"
# - NPZ clip shape: (C,T,H,W) = (3,16,112,112)
# - Dataset returns [T,C,H,W]
# - MAR cache: JSON mapping clip_path -> [mar_mean, mar_max, mar_std, open_ratio]
#
# Pipeline:
# 0) Mount Drive
# 1) Build/Resume MAR cache (FaceMesh)  ✅ handles (C,T,H,W)
# 2) Train:
#      - TSM-fast baseline (ResNetTemporalAvg) + MAR (primary)
#      - CNN-GRU (ResNet18 + GRU) + MAR (backup)
# 3) Calibrate on GLOBAL_VAL:
#      - Temperature scaling
#      - Threshold selection with specificity constraint
# 4) Save best + final packages
#
# ============================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os, json, random, time
from dataclasses import dataclass
from typing import Dict, Any, Optional, Tuple

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    confusion_matrix, average_precision_score
)

import mediapipe as mp
import torchvision

# ----------------------------
# 0) Config
# ----------------------------
@dataclass
class CFG:
    # Splits
    UTA_TRAIN: str = "/content/drive/MyDrive/ES327_Drowsiness/splits/uta_train.csv"
    UTA_VAL:   str = "/content/drive/MyDrive/ES327_Drowsiness/splits/uta_val.csv"
    YAW_TRAIN: str = "/content/drive/MyDrive/ES327_Drowsiness/splits/yawdd_train.csv"
    YAW_VAL:   str = "/content/drive/MyDrive/ES327_Drowsiness/splits/yawdd_val.csv"

    # Clip roots
    CLIPS_DRIVE_ROOT: str = "/content/drive/MyDrive/ES327_Drowsiness/Clips"
    CLIPS_LOCAL_ROOT: str = "/content/Clips"

    # MAR cache
    MAR_SAVE_DIR: str = "/content/drive/MyDrive/ES327_Drowsiness/outputs_mar_cache"
    MAR_CACHE_PATH: str = "/content/drive/MyDrive/ES327_Drowsiness/outputs_mar_cache/mar_cache.json"
    BUILD_MAR_CACHE: bool = True  # set False if MAR cache already built & non-empty

    # Outputs
    SAVE_DIR: str = "/content/drive/MyDrive/ES327_Drowsiness/final_models_tsm_cnn_mar"

    # Training
    SEED: int = 42
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"
    EPOCHS: int = 6
    BATCH_SIZE: int = 32
    LR: float = 1e-4
    WEIGHT_DECAY: float = 1e-4
    NUM_WORKERS: int = 2

    # Clip format
    EXPECT_T: int = 16
    EXPECT_C: int = 3
    NORMALIZE_0_1: bool = True

    # Balanced sampling
    DS_MIX_UTA: float = 0.5
    DS_MIX_YAW: float = 0.5

    # Calibration
    SPEC_TARGET: float = 0.30

    # MAR settings
    MAR_FRAMES: int = 4
    MAR_THRESHOLD: float = 0.55
    MAR_SAVE_EVERY: int = 200

cfg = CFG()
os.makedirs(cfg.SAVE_DIR, exist_ok=True)
os.makedirs(cfg.MAR_SAVE_DIR, exist_ok=True)

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(cfg.SEED)
print("Device:", cfg.DEVICE)

# ----------------------------
# 1) Helpers
# ----------------------------
def clean_path(p: str) -> str:
    return str(p).replace("\\", "/").replace("\u00a0", " ").strip()

def prefer_local(path: str) -> str:
    p = clean_path(path)
    pref = cfg.CLIPS_DRIVE_ROOT.rstrip("/") + "/"
    if p.startswith(pref):
        rel = p[len(pref):]
        local = os.path.join(cfg.CLIPS_LOCAL_ROOT, rel)
        if os.path.exists(local):
            return local
    return p

def load_split_csv(path: str, default_dataset: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df = df.dropna(subset=["clip_path", "label"]).copy()
    df["clip_path"] = df["clip_path"].map(clean_path)
    df["label"] = df["label"].astype(int)
    if "dataset" not in df.columns:
        df["dataset"] = default_dataset
    df["dataset"] = df["dataset"].astype(str).str.lower()
    if "subject_id" not in df.columns:
        df["subject_id"] = "unknown"
    return df

def sample_indices(T: int, n: int) -> np.ndarray:
    if T <= 0: return np.array([], dtype=int)
    if n <= 1: return np.array([T//2], dtype=int)
    return np.linspace(0, T-1, n).round().astype(int)

# ----------------------------
# 2) MAR cache builder (FaceMesh) - handles NPZ clip shape (C,T,H,W)
# ----------------------------
MOUTH_L, MOUTH_R, MOUTH_U, MOUTH_D = 61, 291, 13, 14

def clip_to_rgb_frames(npz_path: str) -> np.ndarray:
    """
    Load npz['clip'] where clip is (C,T,H,W) = (3,16,112,112)
    Return uint8 RGB frames [T,H,W,3]
    """
    z = np.load(npz_path, allow_pickle=True)
    x = z["clip"] if "clip" in z else z[z.files[0]]

    if x.ndim != 4:
        raise ValueError(f"Unexpected ndim {x.ndim} keys={z.files}")

    # (C,T,H,W) -> (T,H,W,C)
    if x.shape[0] == 3:
        x = np.transpose(x, (1, 2, 3, 0))
    # (T,C,H,W) -> (T,H,W,C)
    elif x.shape[1] == 3:
        x = np.transpose(x, (0, 2, 3, 1))
    # already (T,H,W,C)
    elif x.shape[-1] == 3:
        pass
    else:
        raise ValueError(f"Unknown layout {x.shape}")

    x = x.astype(np.float32)
    if x.max() <= 1.5:
        x *= 255.0
    x = np.clip(x, 0, 255).astype(np.uint8)
    return x

def mar_from_landmarks(lms, h: int, w: int) -> float:
    def pt(i):
        lm = lms[i]
        return np.array([lm.x * w, lm.y * h], dtype=np.float32)
    L = pt(MOUTH_L); R = pt(MOUTH_R); U = pt(MOUTH_U); D = pt(MOUTH_D)
    horiz = np.linalg.norm(L - R) + 1e-6
    vert  = np.linalg.norm(U - D)
    return float(vert / horiz)

def compute_mar_features(frames_rgb: np.ndarray, face_mesh) -> list:
    T, H, W, _ = frames_rgb.shape
    idxs = sample_indices(T, cfg.MAR_FRAMES)
    mars = []
    for i in idxs:
        res = face_mesh.process(frames_rgb[i])  # RGB
        if not res.multi_face_landmarks:
            continue
        lms = res.multi_face_landmarks[0].landmark
        mars.append(mar_from_landmarks(lms, H, W))

    if len(mars) == 0:
        return [0.0, 0.0, 0.0, 0.0]

    mars = np.array(mars, dtype=np.float32)
    return [
        float(mars.mean()),
        float(mars.max()),
        float(mars.std()),
        float((mars > cfg.MAR_THRESHOLD).mean())
    ]

def build_mar_cache_if_needed(cfg: CFG):
    if not cfg.BUILD_MAR_CACHE:
        print("Skipping MAR cache build.")
        return

    dfs = []
    for s, dset in [(cfg.UTA_TRAIN,"uta"), (cfg.UTA_VAL,"uta"), (cfg.YAW_TRAIN,"yawdd"), (cfg.YAW_VAL,"yawdd")]:
        if os.path.exists(s):
            df = pd.read_csv(s).dropna(subset=["clip_path","label"]).copy()
            df["clip_path"] = df["clip_path"].map(clean_path)
            dfs.append(df)
    all_df = pd.concat(dfs, ignore_index=True)
    clip_paths = all_df["clip_path"].unique().tolist()

    if os.path.exists(cfg.MAR_CACHE_PATH):
        try:
            mar_cache = json.load(open(cfg.MAR_CACHE_PATH))
            mar_cache = {clean_path(k): v for k,v in mar_cache.items()}
        except Exception:
            mar_cache = {}
    else:
        mar_cache = {}

    print("Unique clips:", len(clip_paths))
    print("Existing MAR entries:", len(mar_cache))
    print("Saving to:", cfg.MAR_CACHE_PATH)

    mp_face_mesh = mp.solutions.face_mesh
    added, no_face, errors = 0, 0, 0
    t0 = time.time()

    with mp_face_mesh.FaceMesh(
        static_image_mode=True,
        max_num_faces=1,
        refine_landmarks=False,
        min_detection_confidence=0.5,
    ) as face_mesh:

        for p in tqdm(clip_paths):
            key = clean_path(p)
            if key in mar_cache:
                continue

            npz_path = prefer_local(key)
            if not os.path.exists(npz_path) and os.path.exists(key):
                npz_path = key

            try:
                frames = clip_to_rgb_frames(npz_path)
                feats = compute_mar_features(frames, face_mesh)
                if feats == [0.0, 0.0, 0.0, 0.0]:
                    no_face += 1
                mar_cache[key] = feats
                added += 1

                if added % cfg.MAR_SAVE_EVERY == 0:
                    json.dump(mar_cache, open(cfg.MAR_CACHE_PATH, "w"))

            except Exception:
                errors += 1
                if errors <= 3:
                    import traceback
                    print("First error on:", npz_path)
                    traceback.print_exc()
                continue

    json.dump(mar_cache, open(cfg.MAR_CACHE_PATH, "w"))
    print("\n✅ MAR cache done")
    print("Total entries:", len(mar_cache))
    print("Newly added:", added)
    print("No-face:", no_face)
    print("Errors:", errors)
    print(f"Time: {(time.time()-t0)/60:.1f} min")

def load_mar_cache(path: str) -> Dict[str, Any]:
    d = json.load(open(path))
    return {clean_path(k): v for k,v in d.items()}

# ----------------------------
# 3) Clip loader for training: NPZ clip (C,T,H,W) -> [T,C,H,W]
# ----------------------------
def load_npz_clip_tchw(npz_path: str) -> np.ndarray:
    z = np.load(npz_path, allow_pickle=True)
    x = z["clip"] if "clip" in z else z[z.files[0]]  # (C,T,H,W)
    x = x.astype(np.float32)

    # (C,T,H,W) -> (T,C,H,W)
    if x.ndim == 4 and x.shape[0] == 3:
        x = np.transpose(x, (1,0,2,3))

    if cfg.NORMALIZE_0_1 and x.max() > 1.5:
        x = x / 255.0

    # enforce T
    T = x.shape[0]
    if T < cfg.EXPECT_T:
        pad = np.repeat(x[-1:], cfg.EXPECT_T - T, axis=0)
        x = np.concatenate([x, pad], axis=0)
    elif T > cfg.EXPECT_T:
        x = x[:cfg.EXPECT_T]

    return x  # [T,C,H,W]

# ----------------------------
# 4) Dataset + sampler
# ----------------------------
class NPZClipsWithMAR(Dataset):
    def __init__(self, df: pd.DataFrame, mar_cache: Dict[str, Any], strict_mar: bool = False):
        self.df = df.reset_index(drop=True)
        self.cache = mar_cache
        self.strict = strict_mar

        self.mar_dim = None
        for i in range(len(self.df)):
            k = clean_path(self.df.loc[i, "clip_path"])
            if k in self.cache:
                self.mar_dim = len(self.cache[k])
                break
        if self.mar_dim is None:
            raise RuntimeError("No MAR features found for any clip_path. Cache empty or mismatch.")

    def __len__(self): return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        clip_path = clean_path(row["clip_path"])
        y = int(row["label"])
        ds = str(row["dataset"])

        npz_path = prefer_local(clip_path)
        if not os.path.exists(npz_path) and os.path.exists(clip_path):
            npz_path = clip_path

        x = load_npz_clip_tchw(npz_path)
        marf = self.cache.get(clip_path, None)

        if marf is None:
            if self.strict:
                raise KeyError(f"Missing MAR: {clip_path}")
            marf = [0.0] * self.mar_dim

        return (
            torch.from_numpy(x),                      # [T,C,H,W]
            torch.tensor(marf, dtype=torch.float32),   # [F]
            torch.tensor(y, dtype=torch.long),
            ds,
            clip_path
        )

def make_balanced_sampler(df: pd.DataFrame, mix: Dict[str, float]) -> WeightedRandomSampler:
    ds = df["dataset"].astype(str).str.lower().values
    counts = pd.Series(ds).value_counts().to_dict()
    w = np.zeros(len(df), dtype=np.float64)
    for i, d in enumerate(ds):
        p = float(mix.get(d, 0.0))
        c = float(counts.get(d, 0))
        w[i] = (p / c) if (p > 0 and c > 0) else 0.0
    return WeightedRandomSampler(torch.as_tensor(w, dtype=torch.double), num_samples=len(df), replacement=True)

# ----------------------------
# 5) YOUR BUILDERS (PASTED)
# ----------------------------
NUM_CLASSES = 2

def build_r3d18(num_classes=2):
    m = torchvision.models.video.r3d_18(weights=None)
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m

class ResNetGRU(nn.Module):
    def __init__(self, hidden=128, num_classes=2):
        super().__init__()
        base = torchvision.models.resnet18(weights=None)
        self.backbone = nn.Sequential(*list(base.children())[:-1])
        self.feat_dim = 512
        self.gru = nn.GRU(self.feat_dim, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, num_classes)

    def forward(self, x):  # (B,T,C,H,W)
        B,T,C,H,W = x.shape
        x = x.reshape(B*T, C, H, W)
        f = self.backbone(x).flatten(1)
        f = f.reshape(B, T, self.feat_dim)
        out, _ = self.gru(f)
        return self.fc(out[:, -1, :])

def build_cnn_gru(num_classes=2):
    return ResNetGRU(hidden=128, num_classes=num_classes)

class ResNetTemporalAvg(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        base = torchvision.models.resnet18(weights=None)
        self.backbone = nn.Sequential(*list(base.children())[:-1])
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):  # (B,C,T,H,W)
        B,C,T,H,W = x.shape
        x = x.permute(0,2,1,3,4).contiguous().view(B*T, C, H, W)
        f = self.backbone(x).flatten(1).view(B, T, 512).mean(dim=1)
        return self.fc(f)

def build_tsm_fast(num_classes=2):
    return ResNetTemporalAvg(num_classes=num_classes)

print("✅ Builders ready.")

# ----------------------------
# 6) Wrappers: fuse MAR with logits/features
# ----------------------------
class TSMWithMAR(nn.Module):
    def __init__(self, backbone: nn.Module, mar_dim: int, num_classes: int = 2):
        super().__init__()
        self.backbone = backbone

        feat_dim = num_classes  # backbone outputs logits [B,2] here
        self.fuse = nn.Sequential(
            nn.Linear(feat_dim + mar_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )

    def forward(self, x_tchw: torch.Tensor, marf: torch.Tensor) -> torch.Tensor:
        x_bcthw = x_tchw.permute(0, 2, 1, 3, 4).contiguous()
        out = self.backbone(x_bcthw)  # logits [B,2]
        fused = torch.cat([out, marf], dim=1)
        return self.fuse(fused)

class CNNGRUWithMAR(nn.Module):
    def __init__(self, backbone: nn.Module, mar_dim: int, num_classes: int = 2):
        super().__init__()
        self.backbone = backbone
        feat_dim = num_classes  # logits [B,2]
        self.fuse = nn.Sequential(
            nn.Linear(feat_dim + mar_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )

    def forward(self, x_tchw: torch.Tensor, marf: torch.Tensor) -> torch.Tensor:
        out = self.backbone(x_tchw)  # logits [B,2]
        fused = torch.cat([out, marf], dim=1)
        return self.fuse(fused)

# ----------------------------
# 7) Inference + metrics
# ----------------------------
@torch.no_grad()
def infer_logits(model: nn.Module, loader: DataLoader, device: str) -> Dict[str, Any]:
    model.eval()
    all_logits, all_y = [], []
    from tqdm import tqdm

    for x, marf, y, ds, path in tqdm(train_loader, desc=f"{family_name} train ep{epoch}", leave=False):
        x = x.to(device, non_blocking=True)
        marf = marf.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = model(x, marf)
        all_logits.append(logits.detach().cpu())
        all_y.append(y.detach().cpu())
    return {"logits": torch.cat(all_logits, dim=0), "y": torch.cat(all_y, dim=0)}

def metrics_from_probs(y_true: np.ndarray, prob_pos: np.ndarray, thr: float) -> Dict[str, float]:
    y_pred = (prob_pos >= thr).astype(np.int32)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    acc = accuracy_score(y_true, y_pred)
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    pr_auc = average_precision_score(y_true, prob_pos)
    return {"F1": float(f1), "PR_AUC": float(pr_auc), "Accuracy": float(acc),
            "Precision": float(precision), "Recall": float(recall), "Specificity": float(specificity),
            "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn)}

# ----------------------------
# 8) Temperature scaling + thresholding
# ----------------------------
class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.log_T = nn.Parameter(torch.zeros(()))

    def forward(self, logits: torch.Tensor) -> torch.Tensor:
        T = torch.exp(self.log_T).clamp(1e-3, 1e3)
        return logits / T

def fit_temperature(logits: torch.Tensor, y: torch.Tensor, device: str) -> float:
    scaler = TemperatureScaler().to(device)
    logits = logits.to(device)
    y = y.to(device)
    optimizer = optim.LBFGS(scaler.parameters(), lr=0.1, max_iter=50, line_search_fn="strong_wolfe")
    loss_fn = nn.CrossEntropyLoss()

    def closure():
        optimizer.zero_grad(set_to_none=True)
        loss = loss_fn(scaler(logits), y)
        loss.backward()
        return loss

    optimizer.step(closure)
    return float(torch.exp(scaler.log_T).detach().cpu().item())

def pick_threshold_with_specificity(y_true: np.ndarray, prob_pos: np.ndarray, spec_target: float) -> Tuple[float, Dict[str, float]]:
    best_thr, best_m, best_f1 = 0.5, None, -1.0
    for thr in np.linspace(0, 1, 501):
        m = metrics_from_probs(y_true, prob_pos, float(thr))
        if m["Specificity"] + 1e-12 >= spec_target and m["F1"] > best_f1:
            best_thr, best_m, best_f1 = float(thr), m, m["F1"]
    if best_m is None:
        for thr in np.linspace(0, 1, 501):
            m = metrics_from_probs(y_true, prob_pos, float(thr))
            if m["F1"] > best_f1:
                best_thr, best_m, best_f1 = float(thr), m, m["F1"]
    return best_thr, best_m

# ----------------------------
# 9) Train + package
# ----------------------------
def train_model_family(family_name: str, build_backbone_fn, wrapper_cls, cfg: CFG,
                       train_df: pd.DataFrame, val_df: pd.DataFrame, mar_cache: Dict[str, Any]) -> Dict[str, Any]:
    print(f"\n==============================\nTraining: {family_name}\n==============================")

    train_ds = NPZClipsWithMAR(train_df, mar_cache, strict_mar=False)
    val_ds   = NPZClipsWithMAR(val_df, mar_cache, strict_mar=False)

    sampler = make_balanced_sampler(train_df, {"uta": cfg.DS_MIX_UTA, "yawdd": cfg.DS_MIX_YAW})
    train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, sampler=sampler,
                              num_workers=cfg.NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=cfg.BATCH_SIZE, shuffle=False,
                            num_workers=cfg.NUM_WORKERS, pin_memory=True)

    mar_dim = train_ds.mar_dim
    backbone = build_backbone_fn(num_classes=2)
    model = wrapper_cls(backbone=backbone, mar_dim=mar_dim, num_classes=2).to(cfg.DEVICE)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=1)

    best_f1 = -1.0
    best_ckpt_path = os.path.join(cfg.SAVE_DIR, f"{family_name}_best.pt")

    for epoch in range(1, cfg.EPOCHS + 1):
        model.train()
        t0 = time.time()
        run_loss, n = 0.0, 0

        # ✅ batch progress bar
        for x, marf, y, ds, path in tqdm(train_loader, desc=f"{family_name} train ep{epoch}", leave=False):
            x = x.to(cfg.DEVICE, non_blocking=True)
            marf = marf.to(cfg.DEVICE, non_blocking=True)
            y = y.to(cfg.DEVICE, non_blocking=True)

            logits = model(x, marf)
            loss = criterion(logits, y)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

            run_loss += float(loss.item()) * x.size(0)
            n += x.size(0)

        # ✅ validation progress bar inside infer
        model.eval()
        all_logits, all_y = [], []
        with torch.no_grad():
            for x, marf, y, ds, path in tqdm(val_loader, desc=f"{family_name} val ep{epoch}", leave=False):
                x = x.to(cfg.DEVICE, non_blocking=True)
                marf = marf.to(cfg.DEVICE, non_blocking=True)
                y = y.to(cfg.DEVICE, non_blocking=True)
                logits = model(x, marf)
                all_logits.append(logits.detach().cpu())
                all_y.append(y.detach().cpu())

        logits = torch.cat(all_logits, dim=0)
        y_true = torch.cat(all_y, dim=0).numpy()
        prob_pos = torch.softmax(logits, dim=1)[:, 1].numpy()
        m = metrics_from_probs(y_true, prob_pos, thr=0.5)

        print(f"[{family_name}] Epoch {epoch}/{cfg.EPOCHS} | loss={run_loss/max(1,n):.4f} | "
              f"VAL F1@0.5={m['F1']:.4f} PR_AUC={m['PR_AUC']:.4f} Spec={m['Specificity']:.4f} Rec={m['Recall']:.4f} | {time.time()-t0:.1f}s")

        scheduler.step(m["F1"])

        if m["F1"] > best_f1:
            best_f1 = m["F1"]
            torch.save({"family": family_name, "model": model.state_dict(), "mar_dim": mar_dim, "cfg": cfg.__dict__}, best_ckpt_path)

    print(f"✅ {family_name}: saved best -> {best_ckpt_path}")
    return {"family": family_name, "best_ckpt_path": best_ckpt_path, "mar_dim": mar_dim, "val_loader": val_loader, "val_df": val_df}

def calibrate_and_package(train_art: Dict[str, Any], build_backbone_fn, wrapper_cls, cfg: CFG) -> Dict[str, Any]:
    family = train_art["family"]
    ckpt_path = train_art["best_ckpt_path"]
    mar_dim = train_art["mar_dim"]
    val_loader = train_art["val_loader"]
    val_df = train_art["val_df"]

    ckpt = torch.load(ckpt_path, map_location=cfg.DEVICE)
    backbone = build_backbone_fn(num_classes=2)
    model = wrapper_cls(backbone=backbone, mar_dim=mar_dim, num_classes=2).to(cfg.DEVICE)
    model.load_state_dict(ckpt["model"], strict=True)
    model.eval()

    out = infer_logits(model, val_loader, cfg.DEVICE)
    logits = out["logits"]  # CPU
    y = out["y"]            # CPU
    y_true = y.numpy()

    T = fit_temperature(logits.clone(), y.clone(), cfg.DEVICE)
    prob_pos = torch.softmax((logits / T), dim=1)[:, 1].numpy()
    thr, m_best = pick_threshold_with_specificity(y_true, prob_pos, cfg.SPEC_TARGET)
    print(f"[{family_name}]  VAL best-thr={thr:.3f} | F1={m_best['F1']:.4f} Spec={m_best['Specificity']:.4f} Rec={m_best['Recall']:.4f}")
    thr, m_plain = pick_threshold_with_specificity(y_true, prob_pos, cfg.SPEC_TARGET)

    print(f"\n[{family}] ✅ Calibration")
    print(f"  T={T:.4f} | thr={thr:.4f} | F1={m_plain['F1']:.4f} Spec={m_plain['Specificity']:.4f} Rec={m_plain['Recall']:.4f} PR_AUC={m_plain['PR_AUC']:.4f}")

    final_path = os.path.join(cfg.SAVE_DIR, f"{family}_final.pt")
    meta_path  = os.path.join(cfg.SAVE_DIR, f"{family}_final_meta.json")

    torch.save({
        "family": family,
        "model": model.state_dict(),
        "mar_dim": mar_dim,
        "temperature_T": T,
        "threshold": thr,
        "spec_target": cfg.SPEC_TARGET,
        "cfg": cfg.__dict__,
    }, final_path)

    with open(meta_path, "w") as f:
        json.dump({
            "family": family,
            "best_ckpt_path": ckpt_path,
            "final_path": final_path,
            "temperature_T": T,
            "threshold": thr,
            "val_metrics_plain": m_plain,
        }, f, indent=2)

    print(f"[{family}] ✅ Saved final -> {final_path}")
    print(f"[{family}] ✅ Saved meta  -> {meta_path}")

    return {"family": family, "final_path": final_path, "T": T, "thr": thr}

# ----------------------------
# 10) MAIN
# ----------------------------
def main(cfg: CFG):
    # Build MAR cache if needed
    build_mar_cache_if_needed(cfg)

    # Load MAR cache
    mar_cache = load_mar_cache(cfg.MAR_CACHE_PATH)
    print("Loaded MAR entries:", len(mar_cache))
    if len(mar_cache) == 0:
        raise RuntimeError("MAR cache is empty. Build step failed.")

    # Load splits
    uta_tr = load_split_csv(cfg.UTA_TRAIN, "uta")
    uta_va = load_split_csv(cfg.UTA_VAL,   "uta")
    yaw_tr = load_split_csv(cfg.YAW_TRAIN, "yawdd")
    yaw_va = load_split_csv(cfg.YAW_VAL,   "yawdd")

    train_df = pd.concat([uta_tr, yaw_tr], ignore_index=True)
    val_df   = pd.concat([uta_va, yaw_va], ignore_index=True)

    # Train both models
    tsm_art = train_model_family("tsm_fast_mar", build_tsm_fast, TSMWithMAR, cfg, train_df, val_df, mar_cache)
    cnn_art = train_model_family("cnn_gru_mar",  build_cnn_gru,  CNNGRUWithMAR, cfg, train_df, val_df, mar_cache)

    # Calibrate + package
    _ = calibrate_and_package(tsm_art, build_tsm_fast, TSMWithMAR, cfg)
    _ = calibrate_and_package(cnn_art, build_cnn_gru,  CNNGRUWithMAR, cfg)

main(cfg)

---
## 5. Calibration & Final Packaging

Fits temperature scaling and selects the decision threshold on the global validation set (no test leakage), then packages each model into its final checkpoint `*_final.pt` with the frozen threshold and temperature baked in. The CNN-GRU F1-selected threshold is τ≈0.001 (Table 3.4).

In [ ]:
# ============================
# FIX infer_logits + RUN ONLY POST-TRAIN STEPS
# - Replaces buggy infer_logits (no family_name/epoch/train_loader references)
# - Loads your existing *_best.pt checkpoints
# - Rebuilds GLOBAL_VAL loader
# - Runs calibration (temperature scaling + threshold pick)
# - Saves *_final.pt + *_final_meta.json for BOTH models
#
# Assumes these already exist in your notebook/session:
#   - cfg (with SAVE_DIR, MAR_CACHE_PATH, DEVICE, etc.)
#   - load_split_csv, load_mar_cache, NPZClipsWithMAR
#   - build_tsm_fast, build_cnn_gru
#   - TSMWithMAR, CNNGRUWithMAR
#   - fit_temperature, pick_threshold_with_specificity, metrics_from_probs
#   - calibrate_and_package (optional; we include a safe version below)
# ============================

import os, json
import torch
from torch.utils.data import DataLoader
import pandas as pd
import numpy as np
import torch.nn as nn
import torch.optim as optim

# ---------- 1) FIX infer_logits ----------
@torch.no_grad()
def infer_logits(model: nn.Module, loader: DataLoader, device: str, desc: str = "infer"):
    from tqdm import tqdm
    model.eval()
    all_logits, all_y = [], []
    for x, marf, y, ds, path in tqdm(loader, desc=desc, leave=False):
        x = x.to(device, non_blocking=True)
        marf = marf.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = model(x, marf)
        all_logits.append(logits.detach().cpu())
        all_y.append(y.detach().cpu())
    return {"logits": torch.cat(all_logits, dim=0), "y": torch.cat(all_y, dim=0)}

# ---------- 2) SAFER calibrate_and_package (uses fixed infer_logits) ----------
def calibrate_and_package(train_art, build_backbone_fn, wrapper_cls, cfg):
    family = train_art["family"]
    ckpt_path = train_art["best_ckpt_path"]
    mar_dim = train_art["mar_dim"]
    val_loader = train_art["val_loader"]

    ckpt = torch.load(ckpt_path, map_location=cfg.DEVICE)

    backbone = build_backbone_fn(num_classes=2)
    model = wrapper_cls(backbone=backbone, mar_dim=mar_dim, num_classes=2).to(cfg.DEVICE)
    model.load_state_dict(ckpt["model"], strict=True)
    model.eval()

    out = infer_logits(model, val_loader, cfg.DEVICE, desc=f"{family} calibrate")
    logits = out["logits"]  # CPU
    y = out["y"]            # CPU
    y_true = y.numpy()

    # temperature
    T = fit_temperature(logits.clone(), y.clone(), cfg.DEVICE)

    # probs after temperature
    prob_pos = torch.softmax((logits / T), dim=1)[:, 1].numpy()

    # threshold selection w/ specificity constraint
    thr, m_plain = pick_threshold_with_specificity(y_true, prob_pos, 0.7)

    print(f"\n[{family}] ✅ Calibration")
    print(f"  T={T:.4f} | thr={thr:.4f} | "
          f"F1={m_plain['F1']:.4f} Spec={m_plain['Specificity']:.4f} "
          f"Rec={m_plain['Recall']:.4f} PR_AUC={m_plain['PR_AUC']:.4f}")

    final_path = os.path.join(cfg.SAVE_DIR, f"{family}_final.pt")
    meta_path  = os.path.join(cfg.SAVE_DIR, f"{family}_final_meta.json")

    torch.save({
        "family": family,
        "model": model.state_dict(),
        "mar_dim": mar_dim,
        "temperature_T": T,
        "threshold": thr,
        "spec_target": 0.7,
        "cfg": cfg.__dict__,
    }, final_path)

    with open(meta_path, "w") as f:
        json.dump({
            "family": family,
            "best_ckpt_path": ckpt_path,
            "final_path": final_path,
            "temperature_T": T,
            "threshold": thr,
            "val_metrics_plain": m_plain,
        }, f, indent=2)

    print(f"[{family}] ✅ Saved final -> {final_path}")
    print(f"[{family}] ✅ Saved meta  -> {meta_path}")

    return {"family": family, "final_path": final_path, "T": T, "thr": thr}

# ---------- 3) BUILD GLOBAL_VAL LOADER ----------
mar_cache = load_mar_cache(cfg.MAR_CACHE_PATH)
uta_va = load_split_csv(cfg.UTA_VAL, "uta")
yaw_va = load_split_csv(cfg.YAW_VAL, "yawdd")
val_df = pd.concat([uta_va, yaw_va], ignore_index=True)

val_ds = NPZClipsWithMAR(val_df, mar_cache, strict_mar=False)
val_loader = DataLoader(
    val_ds,
    batch_size=cfg.BATCH_SIZE,
    shuffle=False,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=True
)

# ---------- 4) CALIBRATE + PACKAGE BOTH MODELS FROM EXISTING BEST CKPTS ----------
tsm_best = os.path.join(cfg.SAVE_DIR, "tsm_fast_mar_best.pt")
cnn_best = os.path.join(cfg.SAVE_DIR, "cnn_gru_mar_best.pt")

assert os.path.exists(tsm_best), f"Missing: {tsm_best}"
assert os.path.exists(cnn_best), f"Missing: {cnn_best}"

tsm_art = {
    "family": "tsm_fast_mar",
    "best_ckpt_path": tsm_best,
    "mar_dim": val_ds.mar_dim,
    "val_loader": val_loader,
}
cnn_art = {
    "family": "cnn_gru_mar",
    "best_ckpt_path": cnn_best,
    "mar_dim": val_ds.mar_dim,
    "val_loader": val_loader,
}

_ = calibrate_and_package(tsm_art, build_tsm_fast, TSMWithMAR, cfg)
_ = calibrate_and_package(cnn_art, build_cnn_gru,  CNNGRUWithMAR, cfg)

print("\n✅ Post-training calibration + final packaging complete.")

---
## 6. Evaluation — CNN-GRU on Test Sets

Loads `cnn_gru_mar_final.pt` (weights + temperature + frozen threshold), evaluates on the YawDD, UTA-RLDD and DROZY test sets, saves per-clip predictions and a summary metrics table (F1, PR-AUC, accuracy, precision, recall, specificity). These are the CNN-GRU numbers in Table 4.2.

In [ ]:
# ============================================================
# CLEAN FINAL EVALUATION SCRIPT (FIXED MAR CACHE MISMATCH)
# - Loads cnn_gru_mar_final.pt (weights + temperature + threshold)
# - Loads MAR cache and matches MAR by "tail after /Clips/"
# - Evaluates on YawDD_test / UTA_test / DROZY_test CSVs
# - Saves per-clip preds + summary table
#
# ✅ Edit ONLY the 3 TEST CSV paths if yours differ.
# ============================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os, json
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    confusion_matrix, average_precision_score
)

import torchvision

# ----------------------------
# 0) PATHS (EDIT THESE 3)
# ----------------------------
YAW_TEST   = "/content/drive/MyDrive/ES327_Drowsiness/splits/yawdd_test.csv"
UTA_TEST   = "/content/drive/MyDrive/ES327_Drowsiness/splits/uta_test.csv"
DROZY_TEST = "/content/drive/MyDrive/ES327_Drowsiness/splits/drozy_test.csv"

FINAL_CKPT     = "/content/drive/MyDrive/ES327_Drowsiness/final_models_tsm_cnn_mar/cnn_gru_mar_final.pt"
MAR_CACHE_PATH = "/content/drive/MyDrive/ES327_Drowsiness/outputs_mar_cache/mar_cache.json"

OUT_DIR = "/content/drive/MyDrive/ES327_Drowsiness/final_models_tsm_cnn_mar/test_eval"
os.makedirs(OUT_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 64
NUM_WORKERS = 2

# ----------------------------
# 1) HELPERS
# ----------------------------
def clean_path(p: str) -> str:
    return str(p).replace("\\","/").replace("\u00a0"," ").strip()

def clip_tail(p: str) -> str:
    """
    Stable key to match MAR cache entries across drive/local/relative paths.
    Prefer substring after '/Clips/' if present.
    """
    p = clean_path(p)
    tag = "/Clips/"
    if tag in p:
        return p.split(tag, 1)[1]
    parts = p.split("/")
    return "/".join(parts[-4:])  # fallback

def load_split_csv(path: str, default_dataset: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df = df.dropna(subset=["clip_path","label"]).copy()
    df["clip_path"] = df["clip_path"].map(clean_path)
    df["label"] = df["label"].astype(int)
    if "dataset" not in df.columns:
        df["dataset"] = default_dataset
    df["dataset"] = df["dataset"].astype(str).str.lower()
    return df

def load_mar_cache_with_tail_index(path: str):
    raw = json.load(open(path))
    mar_cache = {clean_path(k): v for k,v in raw.items()}
    tail_index = {}
    for k in mar_cache.keys():
        tail_index[clip_tail(k)] = k
    return mar_cache, tail_index

def resolve_npz_path(p: str) -> str:
    """
    Makes sure clip_path exists even if CSV uses local vs drive paths.
    Tries:
      - as-is
      - swap /content/Clips/ <-> /content/drive/.../Clips/
    """
    p = clean_path(p)
    if os.path.exists(p):
        return p

    # If it's a local path, map to drive clips root
    if p.startswith("/content/Clips/"):
        tail = p.split("/content/Clips/", 1)[1]
        cand = "/content/drive/MyDrive/ES327_Drowsiness/Clips/" + tail
        if os.path.exists(cand):
            return cand

    # If it's a drive path, map to local
    tag = "/content/drive/MyDrive/ES327_Drowsiness/Clips/"
    if p.startswith(tag):
        tail = p.split(tag, 1)[1]
        cand = "/content/Clips/" + tail
        if os.path.exists(cand):
            return cand

    return p  # will error later if truly missing

def load_npz_clip_tchw(npz_path: str) -> np.ndarray:
    """
    NPZ 'clip' is (C,T,H,W) -> return [T,C,H,W] float32 0..1
    """
    npz_path = resolve_npz_path(npz_path)
    z = np.load(npz_path, allow_pickle=True)
    x = z["clip"] if "clip" in z else z[z.files[0]]
    x = x.astype(np.float32)

    # (C,T,H,W)->(T,C,H,W)
    if x.ndim == 4 and x.shape[0] == 3:
        x = np.transpose(x, (1,0,2,3))

    if x.max() > 1.5:
        x = x / 255.0
    return x

# ----------------------------
# 2) MODEL (your CNN-GRU + MAR fusion)
# ----------------------------
class ResNetGRU(nn.Module):
    def __init__(self, hidden=128, num_classes=2):
        super().__init__()
        base = torchvision.models.resnet18(weights=None)
        self.backbone = nn.Sequential(*list(base.children())[:-1])
        self.feat_dim = 512
        self.gru = nn.GRU(self.feat_dim, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, num_classes)

    def forward(self, x):  # (B,T,C,H,W)
        B,T,C,H,W = x.shape
        x = x.reshape(B*T, C, H, W)
        f = self.backbone(x).flatten(1)
        f = f.reshape(B, T, self.feat_dim)
        out, _ = self.gru(f)
        return self.fc(out[:, -1, :])

def build_cnn_gru(num_classes=2):
    return ResNetGRU(hidden=128, num_classes=num_classes)

class CNNGRUWithMAR(nn.Module):
    def __init__(self, backbone: nn.Module, mar_dim: int, num_classes: int = 2):
        super().__init__()
        self.backbone = backbone
        self.fuse = nn.Sequential(
            nn.Linear(num_classes + mar_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )

    def forward(self, x_tchw: torch.Tensor, marf: torch.Tensor) -> torch.Tensor:
        logits = self.backbone(x_tchw)          # [B,2]
        fused  = torch.cat([logits, marf], 1)   # [B,2+F]
        return self.fuse(fused)                 # [B,2]

# ----------------------------
# 3) DATASET
# ----------------------------
class NPZTestWithMAR(Dataset):
    def __init__(self, df: pd.DataFrame, mar_cache: dict, mar_tail_index: dict):
        self.df = df.reset_index(drop=True)
        self.mar_cache = mar_cache
        self.tail_index = mar_tail_index

        # infer mar_dim by matching tails
        self.mar_dim = None
        for i in range(min(len(self.df), 500)):
            p = clean_path(self.df.loc[i, "clip_path"])
            k = self.tail_index.get(clip_tail(p), None)
            if k is not None:
                self.mar_dim = len(self.mar_cache[k])
                break
        if self.mar_dim is None:
            sample = clean_path(self.df.loc[0,"clip_path"])
            print("Sample test clip_path:", sample)
            print("Sample tail:", clip_tail(sample))
            raise RuntimeError("No MAR entries found for this split (cache mismatch).")

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        p = clean_path(row["clip_path"])
        y = int(row["label"])

        x = load_npz_clip_tchw(p)

        k = self.tail_index.get(clip_tail(p), None)
        marf = self.mar_cache[k] if (k is not None) else [0.0]*self.mar_dim

        return (
            torch.from_numpy(x),                       # [T,C,H,W]
            torch.tensor(marf, dtype=torch.float32),    # [F]
            torch.tensor(y, dtype=torch.long),
            p
        )

# ----------------------------
# 4) METRICS
# ----------------------------
def metrics_from_probs(y_true: np.ndarray, prob_pos: np.ndarray, thr: float):
    y_pred = (prob_pos >= thr).astype(np.int32)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    acc = accuracy_score(y_true, y_pred)
    specificity = tn/(tn+fp) if (tn+fp)>0 else 0.0
    pr_auc = average_precision_score(y_true, prob_pos)
    return {
        "F1": float(f1), "PR_AUC": float(pr_auc), "Accuracy": float(acc),
        "Precision": float(precision), "Recall": float(recall), "Specificity": float(specificity),
        "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn)
    }

# ----------------------------
# 5) LOAD FINAL MODEL
# ----------------------------
ckpt = torch.load(FINAL_CKPT, map_location=DEVICE)
T = float(ckpt["temperature_T"])
THR = float(ckpt["threshold"])
mar_dim = int(ckpt["mar_dim"])

print("Loaded final:", FINAL_CKPT)
print("T =", T)
print("thr =", THR)
print("mar_dim =", mar_dim)

mar_cache, mar_tail_index = load_mar_cache_with_tail_index(MAR_CACHE_PATH)
print("MAR cache entries:", len(mar_cache))
print("Tail index entries:", len(mar_tail_index))

backbone = build_cnn_gru(num_classes=2)
model = CNNGRUWithMAR(backbone=backbone, mar_dim=mar_dim, num_classes=2).to(DEVICE)
model.load_state_dict(ckpt["model"], strict=True)
model.eval()

# ----------------------------
# 6) EVAL ONE SPLIT
# ----------------------------
@torch.no_grad()
def eval_split(name: str, csv_path: str, default_dataset: str):
    df = load_split_csv(csv_path, default_dataset)
    ds = NPZTestWithMAR(df, mar_cache, mar_tail_index)

    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

    all_prob, all_y, all_path = [], [], []

    for x, marf, y, p in tqdm(loader, desc=f"TEST {name}"):
        x = x.to(DEVICE, non_blocking=True)
        marf = marf.to(DEVICE, non_blocking=True)

        logits = model(x, marf)
        logits = logits / T
        prob_pos = torch.softmax(logits, dim=1)[:, 1]

        all_prob.append(prob_pos.detach().cpu().numpy())
        all_y.append(y.numpy())
        all_path += list(p)

    prob = np.concatenate(all_prob, axis=0)
    y_true = np.concatenate(all_y, axis=0)
    m = metrics_from_probs(y_true, prob, THR)

    out_csv = os.path.join(OUT_DIR, f"preds_cnn_gru_mar_{name}.csv")
    pd.DataFrame({
        "clip_path": all_path,
        "y_true": y_true,
        "prob_pos": prob,
        "y_pred": (prob >= THR).astype(int)
    }).to_csv(out_csv, index=False)

    print(f"\n{name} metrics:", m)
    print("Saved preds:", out_csv)
    return m

# ----------------------------
# 7) RUN ALL TESTS + SAVE SUMMARY
# ----------------------------
results = []
results.append(("YawDD_test",  eval_split("YawDD_test",  YAW_TEST,  "yawdd")))
results.append(("UTA_test",    eval_split("UTA_test",    UTA_TEST,  "uta")))
results.append(("DROZY_test",  eval_split("DROZY_test",  DROZY_TEST,"drozy")))

summary = pd.DataFrame([{ "split": k, **v } for k,v in results])
summary_path = os.path.join(OUT_DIR, "summary_cnn_gru_mar_tests.csv")
summary.to_csv(summary_path, index=False)

print("\n==============================")
print("✅ FINAL TEST SUMMARY")
print(summary[["split","F1","PR_AUC","Accuracy","Precision","Recall","Specificity","TP","FP","TN","FN"]])
print("Saved summary:", summary_path)
print("==============================")